## Selección del caso de estudio

Una vez establecida la coneccion entre aprendizaje por refuerzo, razonamiento secuencial y modelos de lenguaje, el siguiente paso consiste en estudiar un caso concreto de post-entrenamiento. En este trabajo se tomará como caso de estudio una variante moderna de aprendizaje por refuerzo aplicada a modelos de lenguaje grandes: DrGRPO, dentro del marco de reinforcement learning from verifiable rewards (RLVR).

La elección de este caso resulta natural porque reúne, en una sola metodología, varios de los conceptos introducidos en las secciones anteriores. En particular, aparece nuevamente una política parametrizada por un modelo, una colección de respuestas generadas para ciertos prompts, una señal de evaluación externa y un procedimiento de actualización cuyo propósito es favorecer respuestas mejor calificadas.


Por esta razón, antes de entrar en la descripción operativa del pipeline, conviene fijar con claridad qué significa RLVR, cuál es la idea general detrás de GRPO y por qué una variante como DrGRPO resulta relevante para tareas de razonamiento.

## Recompensas verificables y el marco RLVR

En el aprendizaje por refuerzo clsico, la recompensa es la señal numérica que indica qué tan conveniente fue una decisión o una trayectoria. Cuando se trabaja con modelos de lenguaje, esa idea se conserva, pero la recompensa ya no proviene necesariamente de un entorno físico o de un simulador tradicional. En muchos casos, la calidad de una respuesta puede evaluarse mediante reglas externas que determinan si dicha respuesta cumple o no con cierta condición.

Esto conduce al marco de reinforcement learning from verifiable rewards (RLVR). La idea central consiste en generar respuestas con el modelo y luego asignarles una recompensa mediante un procedimiento verificable. Por ejemplo, una respuesta puede recibir una evaluación positiva si: contiene el resultado matemático correcto, respeta un formato solicitado, satisface una condición lógica o simbólica, o coincide con una solución que puede comprobarse automáticamente.

La ventaja conceptual de este enfoque es que la señal de recompensa no depende únicamente de una apreciación subjetiva, sino de un criterio que puede verificarse de manera más directa. 

Desde el punto de vista matemático, el principio sigue siendo mismo: si para un prompt $x$ el modelo genera una respuesta $y$, y existe una función de recompensa $r(x,y),$ entonces el post-entrenamiento puede formularse como el problema de ajustar la política del modelo para favorecer respuestas con mayor recompensa esperada.

## Idea general de GRPO

Una vez que se dispone de una recompensa verificable, hace falta un mecanismo de optimización que utilice esa señal para modificar el comportamiento del modelo. En este contexto aparece GRPO, que puede entenderse, a nivel conceptual, como un método de aprendizaje por refuerzo donde para un mismo prompt se generan varias respuestas y luego estas se comparan entre sí a partir de sus recompensas.

La intuición es la siguiente. Dado un prompt, el modelo produce varias completaciones. Cada una de esas completaciones recibe una evaluación. En lugar de pensar únicamente en una respuesta aislada, el método considera el conjunto de respuestas generadas para ese mismo prompt y usa la información relativa entre ellas para orientar la actualización del modelo.

Visto de esta manera, GRPO conserva la estructura esencial del aprendizaje por refuerzo:

- existe una política parametrizada por el modelo de lenguaje;
- la política genera trayectorias, que aquí son secuencias de tokens;
- cada trayectoria recibe una señal de evaluación;
- y el modelo se actualiza para aumentar la probabilidad de respuestas mejor evaluadas.

Lo importante, por ahora, no es entrar todavía en la derivación técnica completa del algoritmo, sino entender que GRPO pertenece al mismo marco general que se ha venido construyendo: decisiones secuenciales, recompensa y optimización de una política.

## Por qué considerar una variante como DrGRPO

Cuando se diseñan métodos de aprendizaje por refuerzo para modelos de lenguaje, no basta con definir una recompensa y optimizarla de manera ingenua. También es necesario cuidar que el método no introduzca incentivos indeseables. En tareas de razonamiento, esto es especialmente importante, porque una respuesta más larga no necesariamente es una respuesta mejor.

Precisamente en este punto adquiere interés la variante DrGRPOE, de manera resumida, como un ajuste o corrección que evita incentivar respuestas solo por ser más largas. 

Esta observación es relevante desde el punto de vista conceptual. Si un método de optimización termina favoreciendo longitud en lugar de calidad real de razonamiento, entonces el modelo puede aparentar un mejor desempeño sin que exista una mejora genuina en la resolución del problema por tanto, una variante como DrGRPO resulta interesante no solo como detalle técnico, sino como ejemplo de un principio más general: en aprendizaje por refuerzo, el diseño del criterio de actualización importa tanto como la definición de la recompensa.

En consecuencia, el estudio de DrGRPO dentro de este reporte no debe entenderse únicamente como la revisión de una herramienta particular, sino como la exploración de una pregunta más profunda: cómo ajustar un modelo para que mejore en tareas de razonamiento sin introducir sesgos artificiales en el comportamiento generado.

## El papel de VeRL dentro de este caso de estudio

Para llevar estos métodos a una implementación práctica se requiere una infraestructura de software capaz de organizar generación de respuestas, evaluación, actualización del modelo y administración de recursos de cómputo. En el caso de estudio seleccionado, ese papel lo cumple **VeRL**.

Se indica que el trabajo práctico se apoyará en VeRL y que este framework se ejecuta dentro de un contenedor de Docker. También se describen aspectos de infraestructura como el uso de GPU, proveedores de cómputo, verificación de drivers, instalación de Docker y localización de ejemplos relacionados con GRPO dentro del repositorio de VeRL. 

Sin embargo, antes de estudiar esos detalles operativos, conviene fijar la idea central: VeRL no constituye el método de aprendizaje por refuerzo en sí mismo, sino el marco técnico dentro del cual ese método puede implementarse. En otras palabras, DrGRPO representa la lógica de optimización que se desea aplicar, mientras que VeRL proporciona la estructura computacional que permite ejecutar dicha lógica en un entorno realista de entrenamiento.

Una vez introducidos los conceptos de RLVR, GRPO, DrGRPO y VeRL, conviene describir la estructura general del procedimiento que se quiere estudiar. Antes de revisar detalles técnicos de implementación, es importante entender la lógica completa del método como una secuencia ordenada de etapas.

De manera esquemática, el procedimiento general puede describirse así:

1. se parte de un modelo de lenguaje base;
2. se selecciona un conjunto de prompts o problemas;
3. para cada prompt, el modelo genera varias respuestas;
4. cada respuesta recibe una evaluación mediante recompensas verificables;
5. esas evaluaciones se utilizan para comparar respuestas del mismo prompt;
6. a partir de esa información se actualiza la política del modelo;
7. el proceso se repite durante múltiples iteraciones.

Esta estructura muestra con claridad que el método no consiste simplemente en producir texto y observarlo, sino en establecer un ciclo de generación, evaluación y ajuste. Precisamente por eso puede interpretarse como una forma de aprendizaje por refuerzo aplicada al contexto de modelos de lenguaje.

## Modelo base y política inicial

El punto de partida del procedimiento es un modelo de lenguaje ya existente. Esto significa que no se comienza con una política vacía ni con parámetros aleatorios, sino con un sistema que ya ha pasado por una etapa previa de entrenamiento y que, por tanto, ya sabe producir texto coherente.

Desde la perspectiva del aprendizaje por refuerzo, este modelo inicial puede verse como una política parametrizada. Si se denotan sus parámetros por $\theta$, entonces dicha política puede escribirse como

$$
\pi_\theta(a \mid s),
$$

donde $s$ representa el contexto disponible y $a$ representa la siguiente acción posible. En el caso de un modelo de lenguaje, el contexto está formado por el prompt junto con los tokens ya generados, mientras que la acción corresponde al siguiente token que el modelo puede emitir.

Por tanto, el modelo base no solo constituye el punto de partida práctico del procedimiento, sino también la política inicial que después será modificada mediante el proceso de optimización.

Una vez fijado el modelo base, se requiere un conjunto de entradas sobre las cuales se llevará a cabo el proceso de generación y evaluación. Estas entradas reciben el nombre de promtp.

Cada prompt puede entenderse como el punto de arranque de una trayectoria de generación. A partir de él, el modelo comienza a producir una secuencia de tokens hasta construir una respuesta completa. Si el prompt se denota por $x$, y la respuesta generada por el modelo se denota por $y$, entonces el proceso completo puede verse como la producción de una salida condicionada:

$$
y \sim \pi_\theta(\cdot \mid x).
$$

En problemas de razonamiento, los prompts suelen consistir en preguntas, ejercicios, instrucciones o problemas cuya respuesta puede evaluarse de algun modo. La calidad del conjunto de prompts es importante, porque determina sobre qué clase de situaciones aprendera el modelo a comportarse mejor.

En este sentido, los prompts cumplen un papel parecido al de las condiciones iniciales en un problema dinamico: fijan el contexto a partir del cual el modelo desplegara una secuencia de decisiones.

## Generación de múltiples respuestas para un mismo prompt

Una caracteristica central del procedimiento asociado con GRPO es que, para un mismo prompt, no se considera únicamente una respuesta. En cambio, el modelo genera varias respuestas distintas. Esta idea es importante porque permite comparar diferentes trayectorias producidas bajo una misma condición inicial.

Si para un prompt $x$ se generan $m$ respuestas, estas pueden representarse como

$$
y_1, y_2, \dots, y_m.
$$

Cada una de esas respuestas corresponde a una trayectoria distinta de generacion, obtenida a partir de la misma política, pero con variación debida al muestreo. Desde el punto de vista conceptual, esto permite observar no una sola conducta del modelo, sino un pequeño conjunto de conductas alternativas frente al mismo problema.

La utilidad de este paso radica en que, si después se evalúan esas respuestas, puede obtenerse información relativa sobre cuáles parecen mejores y cuáles peores. Esa comparación entre respuestas generadas para el mismo prompt es una de las ideas que vuelve especialmente natural este tipo de método en el contexto del post-entrenamiento para razonamiento.

Después de generar varias respuestas, el siguiente paso consiste en evaluarlas. En el marco RLVR, esa evaluación se realiza mediante una o varias funciones de recompensa que asignan una señal numérica de calidad a cada respuesta.

Si $x$ es el prompt y $y_i$ es una respuesta generada, una recompensa puede escribirse abstractamente como

$$

r(x,y_i).

$$

La interpretación de esta función depende de la tarea. En algunos casos, la recompensa mide si el resultado final es correcto. En otros, puede medir si se respetó un formato, si aparece cierta estructura o si la solución cumple restricciones verificables.

Lo importante aquí es que la respuesta del modelo deja de ser evaluada solo por su fluidez o por su parecido con ejemplos previos, y pasa a ser medida con respecto a un criterio explícito de calidad. Esa es una de las diferencias más importantes entre el simple modelado del lenguaje y el post-entrenamiento basado en recompensas.

## Comparación de respuestas dentro de un mismo grupo

Una vez que se han generado varias respuestas para un mismo prompt y que cada una ha recibido una recompensa, este paso es conceptualmente importante, porque el método no se basa solo en saber si una respuesta fue buena o mala en términos absolutos, sino también en cómo se posiciona con respecto a otras respuestas generadas bajo la misma condisión inicial.

Dicho de forma simple, si varias respuestas compiten entre sí para resolver el mismo problema, entonces la comparación entre sus recompensas permite identificar cuáles fueron relativamente mejores dentro del grupo. Esa información comparativa es precisamente la que se utiliza para orientar la actualización de la política.

Esta idea resulta muy natural en tareas de razonamiento. Frente a un mismo problema, pueden existir varias respuestas incorrectas, varias respuestas parcialmente correctas y quizá una respuesta claramente superior. Compararlas dentro del mismo contexto produce una señal más rica que la simple observación aislada de una sola salida.

Después de la generación y la evaluación, se llega a la etapa de actualización. Aquí es donde el procedimiento adquiere de manera más clara el carácter de aprendizaje por refuerzo.

La política del modelo $\pi_\theta$, se modifica con el propósito de aumentar la probabilidad de respuestas mejor evaluadas y disminuir la probabilidad relativa de respuestas peor evaluadas. En otras palabras, las recompensas obtenidas durante la etapa anterior no se usan solo como una medida descriptiva, sino como una señal para ajustar los parámetros del modelo.

Desde el punto de vista conceptual, esta es la misma idea que aparece en aprendizaje por refuerzo clásico: una política interactúa, recibe retroalimentación y luego se actualiza para mejorar su comportamiento esperado. La diferencia es que aquí la trayectoria está formada por tokens y la recompensa se calcula sobre respuestas generadas por un modelo de lenguaje.

El proceso descrito no ocurre una sola vez. La generación de respuestas, su evaluación y la actualización de la política se repiten durante muchas iteraciones. 

Esta repetición es esencial, porque el aprendizaje por refuerzo no depende de una sola observación, sino de la acumulación progresiva de experiencia y de sucesivas correcciones en la política. En el contexto de modelos de lenguaje, esto significa que el comportamiento deseado no aparece instantáneamente, sino como resultado de un proceso iterativo de refinamiento.

Después de describir la estructura general del método, ya puede verse con claridad por qué este tipo de post-entrenamiento pertenece al marco del aprendizaje por refuerzo.

- El modelo de lenguaje actúa como una política parametrizada.
- La generación de una respuesta constituye una trayectoria secuencial.
- Las funciones de recompensa proporcionan una señal de evaluación.
- La política se actualiza para favorecer trayectorias mejor evaluadas.

Por consiguiente, aunque el entorno ya no sea un tablero, un robot o un simulador físico, la estructura matemática básica permanece: existe una política, existen decisiones secuenciales, existe una recompensa y existe una regla de optimización guiada por esa recompensa.

Esto permite entender el post-entrenamiento para razonamiento no como una ruptura con el aprendizaje por refuerzo clásico, sino como una extensión de sus principios al caso de modelos de lenguaje grandes.